In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image

warnings.filterwarnings("ignore")
tqdm.pandas()

In [2]:
DEVICE = "mps" if torch.mps.is_available() else "cpu"

DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(DEVICE)

mps


In [3]:
res_net50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(DEVICE)

In [4]:
res_net18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE)

In [6]:
res_net34 = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1).to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /Users/semyonkuricyn/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:16<00:00, 5.37MB/s]


In [7]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

# val_df = train_df.head(500)

In [98]:
class ImbaDataset(Dataset):
    def __init__(self, root_dir, df):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        w, h = img.size if isinstance(img, Image.Image) else (img.shape[2], img.shape[1])
        min_side = min(w, h)
        img_cropped = transforms.CenterCrop(min_side)(img)
        img_resized = transforms.Resize((256, 256))(img_cropped)
        img_tensor = transforms.ToTensor()(img_resized)
        
        return img_tensor

In [99]:
tr_images_ds = ImbaDataset(IMAGE_FOLDER, train_df)

In [72]:
batch_size = 32

images = DataLoader(tr_images_ds, batch_size=batch_size, shuffle=False)

In [74]:
sureness50 = []
# sureness18 = []
# sureness34 = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure50 = res_net50(batch.to(DEVICE))
        # image_sure18 = res_net18(batch.to(DEVICE))
        # image_sure34 = res_net34(batch.to(DEVICE))

    sure50 = image_sure50.to('cpu').numpy()
    # sure18 = image_sure18.to('cpu').numpy()
    # sure34 = image_sure34.to('cpu').numpy()

    sureness50.extend(sure50)
    # sureness18.extend(sure18)
    # sureness34.extend(sure34)

100%|██████████| 1065/1065 [07:06<00:00,  2.50it/s]


In [75]:
def dif_2max(lst):
    lst1 = [np.exp(-i) for i in lst]
    sm = sum(lst1)
    lst1 = [i / sm for i in lst1]
    l = lst1[::]
    l[l.index(max(l))] = 0
    return float(max(lst1) - max(l))

In [76]:
obj50 = [dif_2max(e) for e in sureness50]
# obj18 = [dif_2max(e) for e in sureness18]
# obj34 = [dif_2max(e) for e in sureness34]

labels = train_df['label'].to_numpy().tolist()

In [77]:
from sklearn.metrics import roc_auc_score

In [78]:
score50 = roc_auc_score(labels, obj50)
# score18 = roc_auc_score(labels, obj18)
# score34 = roc_auc_score(labels, obj34)

In [79]:
print(score50)

0.48783205674187474


In [80]:
train_df['obj50'] = obj50
# train_df['obj34'] = obj34
# train_df['obj18'] = obj18

In [81]:
groups = train_df['card_identifier_id'].unique()

In [82]:
train_lst = train_df.to_numpy().tolist()

In [84]:
train_lst[0]

[0,
 4035,
 1,
 'Multibook® B6+ Твердый переплет',
 'Представляем Multibook® B6+ — идеальный помощник для планирования и организации! Этот блокнот с твердым переплетом и надежным диск-байндингом (DISK BOUND) гарантирует долговечность и удобство использования. Листья с четкими линиями и специальная закладка помогут вам структурировать задачи, записывать идеи и планировать неделю с максимальной эффективностью. Компактный формат B6+ легко помещается в сумку или рюкзак, делая его незаменимым спутником в поездках, на работе или учебе. Создайте свой уникальный органайзер с Multibook® и почувствуйте разницу в организации своего времени!',
 0.030114486813545227,
 0.015804961323738098,
 0.0038588568568229675]

In [85]:
d = {i: [[], [], [], []] for i in groups}

for id_, card_identifier_id, label, name, description, obj50, obj18, obj34 in train_lst:
    d[card_identifier_id][0].append(float(label))
    d[card_identifier_id][1].append(float(obj50))
    # d[card_identifier_id][2].append(float(obj18))
    # d[card_identifier_id][3].append(float(obj34))

In [86]:
for i in groups:
    print(d[i])

[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.030114486813545227, 0.009588509798049927, 0.021190941333770752, 0.057018619030714035, 0.028499171137809753, 0.06995278596878052, 0.009637214243412018, 0.028726961463689804, 0.020590733736753464, 0.007921412587165833, 0.016421683132648468, 0.001169636845588684], [], []]
[[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0003765970468521118, 0.024869404733181, 0.004839148372411728, 0.016738224774599075, 0.009026501327753067, 0.010491857305169106, 0.01649550534784794, 0.0012404043227434158, 0.004424670711159706], [], []]
[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.02168903686106205, 0.000919930636882782, 0.006729981862008572, 0.055359624326229095, 0.006655452772974968, 0.001501893624663353, 0.010259242728352547, 0.05572108179330826, 9.978748857975006e-05, 0.021137632429599762], [], []]
[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.023338252678513527, 0.010653506964445114, 0.0033601690083

In [87]:
sc50 = 0
# sc18 = 0
# sc34 = 0

ln = 0

for i in groups:
    if len(set(d[i][0])) == 1:
        # d[i][0] += [0, 1]
        # d[i][1] += [0, 1]
        # d[i][2] += [0, 1]
        # d[i][3] += [0, 1]
        pass
    else:
        s50 = roc_auc_score(d[i][0], d[i][1])
        # s18 = roc_auc_score(d[i][0], d[i][2])
        # s34 = roc_auc_score(d[i][0], d[i][3])

        sc50 += s50 * len(d[i][0])
        # sc18 += s18 * len(d[i][0])
        # sc34 += s34 * len(d[i][0])

        ln += len(d[i][0])

sc50 /= ln
# sc18 /= ln
# sc34 /= ln

print(sc50)


0.5011603886569919


In [92]:
for i in range(1000):
    train_df[f"f{i}"] = [sureness50[j][i] for j in range(len(sureness50))]

In [93]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
features = [f"f{i}" for i in range(1000)]
target = 'label'

X_tr, X_val, y_tr, y_val = train_test_split(train_df[features], train_df[target], test_size=0.3, shuffle=True)

In [96]:
from sklearn.metrics import accuracy_score

In [97]:
model = LogisticRegression(n_jobs=-1, solver='newton-cholesky')

model.fit(X_tr, y_tr)

pred = model.predict(X_val)
print(accuracy_score(pred, y_val))

0.8878020150640712


In [101]:
ts_images_ds = ImbaDataset(IMAGE_FOLDER, test_df)

In [ ]:
images_ts = DataLoader(ts_images_ds, batch_size=batch_size, shuffle=False)

In [107]:
sureness50_ts = []

for images_ in tqdm(images_ts):
    with torch.no_grad():
        sure = res_net50(images_.to(DEVICE))

    sureness50_ts.extend(sure.to('cpu').numpy().tolist())

100%|██████████| 264/264 [01:50<00:00,  2.39it/s]


In [123]:
for i in range(1000):
    test_df[f"f{i}"] = [sureness50_ts[j][i] for j in range(len(sureness50_ts))]

pred_ts = model.predict_proba(test_df[features])

In [124]:
test_df['y_pred'] = [i[1] for i in pred_ts.tolist()]

In [125]:
(test_df[["id", 'y_pred']]).to_csv("w_resnet.csv", index=False)